In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd
import numpy as np
import tqdm as tqdm

Mounted at /content/drive


In [12]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")
id_mappings_dir = os.path.join(preprocessed_dir, "id_mappings")
split_features_dir = os.path.join(preprocessed_dir, "split_features")

df_id_path = os.path.join(split_features_dir, "df_id.csv")
df = pd.read_csv(df_id_path)

print(f"Loaded df_id.csv with shape: {df.shape}")
df.head()

Loaded df_id.csv with shape: (3730329, 3)


,user_id_idx,item_id_idx,scene_idx
0,41688,13590,4
1,39834,21351,2
2,22354,33398,2
3,17547,11190,1
4,55873,33398,2


### Loading ID mapping Dictionaries
This section loads the saved ID-to-index mapping dictionaries for:

- `user_id` -> `user_id_idx`
- `item_id` -> `item_id_idx`
- `scene` -> `scene_idx`

These mappings were created during preprocessing and will be reused for encoding and embedding lookup.

In [5]:
# Load mapping dictionaries
def load_mapping(path):
    with open(path, 'r') as f:
        return json.load(f)

In [13]:
# Mapping file paths
user_map_path = os.path.join(id_mappings_dir, "user_id_2idx.json")
item_map_path = os.path.join(id_mappings_dir, "item_id_2idx.json")
scene_map_path = os.path.join(id_mappings_dir, "scene_2idx.json")

In [14]:
# Load mappings
with open(user_map_path, 'r') as f:
    user2idx = json.load(f)

with open(item_map_path, 'r') as f:
    item2idx = json.load(f)

with open(scene_map_path, 'r') as f:
    scene2idx = json.load(f)

print(f"Loaded {len(user2idx)} user IDs, {len(item2idx)} item IDs, and {len(scene2idx)} scenes.")

Loaded 67615 user IDs, 59142 item IDs, and 5 scenes.


In [15]:
# 🔍 Check that all values in df are present in the mappings

missing_users = df[~df['user_id_idx'].isin(user2idx.values())]
missing_items = df[~df['item_id_idx'].isin(item2idx.values())]
missing_scenes = df[~df['scene_idx'].isin(scene2idx.values())]

print(f"Missing user_id indices: {len(missing_users)}")
print(f"Missing item_id indices: {len(missing_items)}")
print(f"Missing scene indices: {len(missing_scenes)}")

Missing user_id indices: 0
Missing item_id indices: 0
Missing scene indices: 0


## One-Hot vs Index vs Embedding Strategy

Before encoding ID-based features, lets understand the different strategies available depending on the type of model and task:

### Index Encoding
- Each unique ID (e.g., `user_id`, `item_id`) is assigned an **integer index**.
- Used as an input to an embedding layer.
- Efficient for deep learning models (low memory footprint).
- Example:
  - `user_id: 123456 → user_id_idx: 932`

### One-Hot Encoding
- Represents each ID as a binary vector with a 1 at the corresponding index position.
- High dimensional and sparse.
- Typically used in linear models or tree-based algorithms.
- Example:
  - `item_id_idx: 2 → [0, 0, 1, 0, 0, ..., 0]`

### Embedding Lookup (Used in ColdFusionX)
- Uses `user_id_idx` and `item_id_idx` as **keys** to lookup a learned dense vector from an embedding matrix.
- This allows the model to learn compact representations of categorical variables.
- Essential in recommendation systems where users/items have latent factors.

### Strategy for ColdFusionX:
- **Index encoding** for `user_id`, `item_id`, and `scene`, which will be fed into **embedding layers** during model training.

## Encode IDs as Indices

We now replace the original categorical values (`user_id`, `item_id`, and `scene`) with their corresponding integer indices using the mapping dictionaries:

- `user_id` → `user_id_idx`
- `item_id` → `item_id_idx`
- `scene` → `scene_idx`

These will later be used as inputs to embedding layers during model training. The mapping ensures consistency across training, validation, and inference stages.

In [17]:
# We'll just ensure the existing *_idx columns are integers
df['user_id_idx'] = df['user_id_idx'].astype(int)
df['item_id_idx'] = df['item_id_idx'].astype(int)
df['scene_idx'] = df['scene_idx'].astype(int)

# Final shape check
print("Index encoding complete.")
df[['user_id_idx', 'item_id_idx', 'scene_idx']].head()

Index encoding complete.


,user_id_idx,item_id_idx,scene_idx
0,41688,13590,4
1,39834,21351,2
2,22354,33398,2
3,17547,11190,1
4,55873,33398,2


In [18]:
# Output paths
df_id_encoded_pkl = os.path.join(split_features_dir, "df_id_encoded.pkl")
df_id_encoded_csv = os.path.join(split_features_dir, "df_id_encoded.csv")

In [19]:
# Saving
df.to_pickle(df_id_encoded_pkl)
df.to_csv(df_id_encoded_csv, index=False)

print(f"Saved encoded ID dataset as:")
print(f"- Pickle -> {df_id_encoded_pkl}")
print(f"- CSV    -> {df_id_encoded_csv}")

Saved encoded ID dataset as:
- Pickle -> /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/split_features/df_id_encoded.pkl
- CSV    -> /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/split_features/df_id_encoded.csv


### Adding Vocabulary Stats for Model Input

The number of unique values for each ID feature (user, item, scene) and save them into a reusable JSON file: `id_vocab_config.json`. This will be helpful for setting up embedding layers during model building.

In [20]:
id_vocab_config = {
    "num_users": df["user_id_idx"].nunique(),
    "num_items": df["item_id_idx"].nunique(),
    "num_scenes": df["scene_idx"].nunique()
}

In [21]:
vocab_config_path = os.path.join(id_mappings_dir, "id_vocab_config.json")

with open(vocab_config_path, 'w') as f:
    json.dump(id_vocab_config, f, indent=4)

print("Saved ID vocabulary config to:", vocab_config_path)
print(json.dumps(id_vocab_config, indent=4))

Saved ID vocabulary config to: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings/id_vocab_config.json
{
    "num_users": 67615,
    "num_items": 59142,
    "num_scenes": 5
}
